
# Libdevice (`tl.extra.libdevice`) function
Triton can invoke a custom function from an external library.
In this example, we will use the `libdevice` library to apply `asin` on a tensor.

Please refer to [CUDA libdevice-users-guide](https://docs.nvidia.com/cuda/libdevice-users-guide/index.html) and/or [HIP device-lib source code](https://github.com/ROCm/llvm-project/tree/amd-staging/amd/device-libs/ocml/src) regarding the semantics of all available libdevice functions.

In `libdevice.py`, we try to aggregate functions with the same computation but different data types together.
For example, both `__nv_asin` and `__nv_asinf` calculate the principal value of the arc sine of the input, but `__nv_asin` operates on `double` and `__nv_asinf` operates on `float`.
Triton automatically selects the correct underlying device function to invoke based on input and output types.


## asin Kernel



In [ ]:
import torch

import triton
import triton.language as tl
import inspect
import os
from triton.language.extra import libdevice

from pathlib import Path

DEVICE = triton.runtime.driver.active.get_active_torch_device()


@triton.jit
def asin_kernel(
    x_ptr,
    y_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    x = libdevice.asin(x)
    tl.store(y_ptr + offsets, x, mask=mask)

## Using the default libdevice library path
We can use the default libdevice library path encoded in `triton/language/math.py`



In [ ]:
torch.manual_seed(0)
size = 98432
x = torch.rand(size, device=DEVICE)
output_triton = torch.zeros(size, device=DEVICE)
output_torch = torch.asin(x)
assert x.is_cuda and output_triton.is_cuda
n_elements = output_torch.numel()
grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']), )
asin_kernel[grid](x, output_triton, n_elements, BLOCK_SIZE=1024)
print(output_torch)
print(output_triton)
print(f'The maximum difference between torch and triton is '
      f'{torch.max(torch.abs(output_torch - output_triton))}')

## Customize the libdevice library path
We can also customize the libdevice library path by passing the path to the `libdevice` library to the `asin` kernel.



In [ ]:
def is_cuda():
    return triton.runtime.driver.active.get_current_target().backend == "cuda"


def is_hip():
    return triton.runtime.driver.active.get_current_target().backend == "hip"


current_file = inspect.getfile(inspect.currentframe())
current_dir = Path(os.path.dirname(os.path.abspath(current_file)))

if is_cuda():
    libdir = current_dir.parent.parent / 'third_party/nvidia/backend/lib'
    extern_libs = {'libdevice': str(libdir / 'libdevice.10.bc')}
elif is_hip():
    libdir = current_dir.parent.parent / 'third_party/amd/backend/lib'
    extern_libs = {}
    libs = ["ocml", "ockl"]
    for lib in libs:
        extern_libs[lib] = str(libdir / f'{lib}.bc')
else:
    raise RuntimeError('unknown backend')

output_triton = torch.empty_like(x)
asin_kernel[grid](x, output_triton, n_elements, BLOCK_SIZE=1024, extern_libs=extern_libs)
print(output_torch)
print(output_triton)
print(f'The maximum difference between torch and triton is '
      f'{torch.max(torch.abs(output_torch - output_triton))}')

# Challenge: External Functions

The cells above are the official Triton tutorial. Now complete the local exercise below without changing the official implementation.

## Challenge

1. Build the familiar masked vector offsets.
2. Load values constrained to asin's real domain `[-1, 1]`.
3. Call `libdevice.asin` inside the JIT kernel.
4. Store the result and validate both float32 and float64.
5. Explain why an explicit external-library path is usually unnecessary.

The exercise uses Triton's default backend library mapping. Supplying custom
bitcode through `extern_libs` is an advanced deployment option and should only
reference reviewed, trusted libraries. The numerical check uses a tighter
tolerance than the lower-precision matrix lessons because both references call
the backend's correctly rounded special-function implementation.

Afterward, inspect other functions exposed by `libdevice`, but keep domain
restrictions and dtype behavior in the host wrapper.

In [ ]:
#!/usr/bin/env python3
"""Day 07 puzzle: call the backend libdevice asin implementation."""

import torch
import triton
import triton.language as tl
from triton.language.extra import libdevice


@triton.jit
def asin_kernel(x_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    # EXERCISE(1): masked-load x, call libdevice.asin, and masked-store output.
    pass


def asin(x: torch.Tensor) -> torch.Tensor:
    raise NotImplementedError("EXERCISE(2): launch asin_kernel")


def run() -> None:
    x = torch.linspace(-1, 1, 100_003, device="cuda")
    torch.testing.assert_close(asin(x), torch.asin(x), atol=2e-6, rtol=2e-6)
    print("Day 07 libdevice asin: correctness OK")

In [ ]:
run()